# Test Metrics Summary

This notebook reads a `test_metrics_over_all.csv` file and prints mean/std per metric, overall and by body part.
Use the utility functions below to load a CSV and display results for multiple models in the same notebook.


In [38]:
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display

METRICS = ["MAE", "MSE", "PSNR", "SSIM"]

def _sanitize_metric_columns(df: pd.DataFrame, metrics=METRICS) -> pd.DataFrame:
    """Convert +/-inf to NaN so mean/std are computed on finite values only."""
    df = df.copy()
    for col in metrics:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")
            df[col] = df[col].replace([np.inf, -np.inf], np.nan)
    return df

def load_metrics_csv(csv_path: Path) -> pd.DataFrame:
    if not csv_path.exists():
        raise FileNotFoundError(f"CSV not found: {csv_path}")

    # The file has a header row: file_name, MAE, MSE, PSNR, SSIM[, mask_voxels]
    # Some files may include a leading unnamed index column; drop it if present.
    df = pd.read_csv(csv_path)

    if "file_name" not in df.columns:
        df = df.iloc[:, 1:]

    if "file_name" not in df.columns:
        raise ValueError("Could not find 'file_name' column after dropping index column.")

    if df.shape[1] == 5:
        df.columns = ["file_name", "MAE", "MSE", "PSNR", "SSIM"]
    elif df.shape[1] == 6:
        df.columns = ["file_name", "MAE", "MSE", "PSNR", "SSIM", "mask_voxels"]

    df = _sanitize_metric_columns(df, METRICS)

    if "mask_voxels" in df.columns:
        df["mask_voxels"] = pd.to_numeric(df["mask_voxels"], errors="coerce")

    return df

def load_volume_metrics_csv(volume_csv_path: Path) -> pd.DataFrame:
    if not volume_csv_path.exists():
        raise FileNotFoundError(f"CSV not found: {volume_csv_path}")

    df = pd.read_csv(volume_csv_path)

    if "volume_id" not in df.columns:
        df = df.iloc[:, 1:]

    if "volume_id" not in df.columns:
        raise ValueError("Could not find 'volume_id' column after dropping index column.")

    df = _sanitize_metric_columns(df, METRICS)

    if "mask_voxels" in df.columns:
        df["mask_voxels"] = pd.to_numeric(df["mask_voxels"], errors="coerce")
    if "slice_count" in df.columns:
        df["slice_count"] = pd.to_numeric(df["slice_count"], errors="coerce")

    return df

def _fmt(v: float) -> str:
    return "n/a" if pd.isna(v) else f"{v:.6g}"

def format_mean_std(summary: pd.DataFrame, mean_col: str, std_col: str, out_name: str) -> pd.DataFrame:
    formatted = summary.apply(
        lambda row: f"{_fmt(row[mean_col])} +- {_fmt(row[std_col])}",
        axis=1,
    ).to_frame(name=out_name)
    return formatted

def overall_mean_std(df: pd.DataFrame) -> pd.DataFrame:
    overall_mean = df[METRICS].mean()
    overall_std = df[METRICS].std()
    overall_summary = pd.DataFrame({"mean": overall_mean, "std": overall_std})
    return format_mean_std(overall_summary, "mean", "std", "mean +- std")

def add_body_part(df: pd.DataFrame, col: str = "file_name") -> pd.DataFrame:
    # Body part grouping derived from filename prefix (e.g., AB_*, HN_*)
    # Example filename: AB_1ABC127-99.nii -> body_part = AB
    df = df.copy()
    df["body_part"] = df[col].astype(str).str.split("_").str[0]
    return df

def mean_std_by_body_part(df: pd.DataFrame) -> pd.DataFrame:
    grouped = df.groupby("body_part")[METRICS].agg(["mean", "std"])

    # Reformat to mean +- std per metric
    formatted = pd.DataFrame(index=grouped.index)
    for metric in METRICS:
        mean_col = (metric, "mean")
        std_col = (metric, "std")
        formatted[metric] = grouped.apply(
            lambda row: f"{_fmt(row[mean_col])} +- {_fmt(row[std_col])}",
            axis=1,
        )
    return formatted

def _print_non_finite_counts(df: pd.DataFrame, title: str) -> None:
    issues = {}
    for metric in METRICS:
        non_finite = (~np.isfinite(df[metric])).sum()
        if non_finite:
            issues[metric] = int(non_finite)
    if issues:
        counts = ", ".join([f"{m}: {c}" for m, c in issues.items()])
        print(f"{title} | non-finite values ignored in stats -> {counts}")

def show_metrics_for_csv(csv_path: Path) -> None:
    df = load_metrics_csv(csv_path)

    _print_non_finite_counts(df, "Slice metrics")
    print("Overall metrics (slice-based):")
    display(overall_mean_std(df))

    df_bp = add_body_part(df, col="file_name")
    print("Metrics by body part (slice-based):")
    display(mean_std_by_body_part(df_bp))

    volume_csv_path = csv_path.parent / "test_metrics_over_volume.csv"
    if volume_csv_path.exists():
        vol_df = load_volume_metrics_csv(volume_csv_path)
        _print_non_finite_counts(vol_df, "Volume metrics")
        print("Overall metrics (volume-based):")
        display(overall_mean_std(vol_df))

        vol_df_bp = add_body_part(vol_df, col="volume_id")
        print("Metrics by body part (volume-based):")
        display(mean_std_by_body_part(vol_df_bp))
    else:
        print(f"Volume metrics CSV not found: {volume_csv_path}")
        print("Run: python training/scripts/compute_volume_metrics.py <path/to/test_metrics_over_all.csv>")


In [39]:
# Per-slice metrics and influence for a single volume
def show_volume_slice_influence(df: pd.DataFrame, volume=None) -> None:
    # volume can be a volume_id string (e.g., 'AB_1ABA009') or an index into the unique volumes list
    if "file_name" not in df.columns:
        raise ValueError("df must contain a 'file_name' column.")

    def _volume_key(name: str) -> str:
        base = name
        if base.endswith(".nii.gz"):
            base = base[:-7]
        elif base.endswith(".nii"):
            base = base[:-4]
        # Use the part before the last '-' as volume id
        return base.rsplit("-", 1)[0]

    def _slice_key(name: str) -> str:
        base = name
        if base.endswith(".nii.gz"):
            base = base[:-7]
        elif base.endswith(".nii"):
            base = base[:-4]
        parts = base.rsplit("-", 1)
        return parts[1] if len(parts) == 2 else ""

    df = df.copy()
    df["volume_id"] = df["file_name"].astype(str).map(_volume_key)
    df["slice_id"] = df["file_name"].astype(str).map(_slice_key)

    volumes = sorted(df["volume_id"].dropna().unique())
    if not volumes:
        print("No volumes found in df.")
        return

    if volume is None:
        volume_id = volumes[0]
    elif isinstance(volume, int):
        if volume < 0 or volume >= len(volumes):
            raise IndexError(f"volume index out of range: {volume} (0..{len(volumes)-1})")
        volume_id = volumes[volume]
    else:
        volume_id = str(volume)
        if volume_id not in volumes:
            raise ValueError(f"volume_id not found: {volume_id}")

    vol_df = df[df["volume_id"] == volume_id].copy()
    if vol_df.empty:
        print(f"No slices found for volume {volume_id}.")
        return

    metrics_cols = ["MAE", "MSE", "PSNR", "SSIM"]
    vol_df[metrics_cols] = vol_df[metrics_cols].apply(pd.to_numeric, errors="coerce")

    if "mask_voxels" not in vol_df.columns:
        print("No mask_voxels column found; cannot compute volume influence.")
        display(vol_df[["file_name", "slice_id", *metrics_cols]])
        return

    vol_df["mask_voxels"] = pd.to_numeric(vol_df["mask_voxels"], errors="coerce").fillna(0)
    total_mask = vol_df["mask_voxels"].sum()
    if total_mask == 0:
        print("mask_voxels sum to 0 for this volume; cannot compute influence.")
        display(vol_df[["file_name", "slice_id", *metrics_cols, "mask_voxels"]])
        return

    vol_df["weight_pct"] = (vol_df["mask_voxels"] / total_mask) * 100

    # Volume-weighted metric for this volume
    weights = vol_df["mask_voxels"] / total_mask
    weighted_mean = vol_df[metrics_cols].mul(weights, axis=0).sum()

    print(f"Volume: {volume_id} (slices={len(vol_df)}, total_mask={int(total_mask)})")
    display(pd.DataFrame({"weighted_mean": weighted_mean}))

    # Per-slice influence table
    show_cols = ["slice_id", *metrics_cols, "mask_voxels", "weight_pct", "file_name"]
    display(vol_df.sort_values("slice_id")[show_cols])

# Example usage (pick a volume index or id)
# show_volume_slice_influence(df, volume=0)
# show_volume_slice_influence(df, volume="AB_1ABA009")


# CUT


## Abdomen


In [40]:
csv_path = Path("/local/scratch/datasets/FullbodySCT/Synthrad_combined_preprocessed/100results/cut_synthrad_abdomen_final/test_50/test_metrics_over_all.csv")
df = load_metrics_csv(csv_path)

# show_volume_slice_influence(df, volume=3)


In [41]:
# Run metrics for the selected CSV path

csv_path = Path("/local/scratch/datasets/FullbodySCT/Synthrad_combined_preprocessed/100results/cut_synthrad_abdomen_final/test_50/test_metrics_over_all.csv")
show_metrics_for_csv(csv_path)


Slice metrics | non-finite values ignored in stats -> MAE: 55, MSE: 55, PSNR: 58, SSIM: 55
Overall metrics (slice-based):


,mean +- std
MAE,118.834 +- 58.5118
MSE,4360.42 +- 1446.23
PSNR,36.402 +- 3.1654
SSIM,0.62013 +- 0.0722848


Metrics by body part (slice-based):


,MAE,MSE,PSNR,SSIM
body_part,,,,
AB,118.834 +- 58.5118,4360.42 +- 1446.23,36.402 +- 3.1654,0.62013 +- 0.0722848


Overall metrics (volume-based):


,mean +- std
MAE,119.202 +- 29.6626
MSE,4518.64 +- 844.631
PSNR,36.0676 +- 0.691402
SSIM,0.619675 +- 0.0309146


Metrics by body part (volume-based):


,MAE,MSE,PSNR,SSIM
body_part,,,,
AB,119.202 +- 29.6626,4518.64 +- 844.631,36.0676 +- 0.691402,0.619675 +- 0.0309146


## Brain


In [42]:
csv_path = Path("/local/scratch/datasets/FullbodySCT/Synthrad_combined_preprocessed/100results/cut_synthrad_brain_final/test_50/test_metrics_over_all.csv")
df = load_metrics_csv(csv_path)

# show_volume_slice_influence(df, volume=3)


In [43]:
# Run metrics for the selected CSV path

csv_path = Path("/local/scratch/datasets/FullbodySCT/Synthrad_combined_preprocessed/100results/cut_synthrad_brain_final/test_50/test_metrics_over_all.csv")
show_metrics_for_csv(csv_path)


Slice metrics | non-finite values ignored in stats -> PSNR: 120
Overall metrics (slice-based):


,mean +- std
MAE,180.717 +- 149.697
MSE,1638.68 +- 839.772
PSNR,40.258 +- 1.71539
SSIM,0.73428 +- 0.156331


Metrics by body part (slice-based):


,MAE,MSE,PSNR,SSIM
body_part,,,,
brain,180.717 +- 149.697,1638.68 +- 839.772,40.258 +- 1.71539,0.73428 +- 0.156331


Overall metrics (volume-based):


,mean +- std
MAE,150.888 +- 28.8315
MSE,1566.74 +- 151.691
PSNR,40.1822 +- 0.485994
SSIM,0.756722 +- 0.0374324


Metrics by body part (volume-based):


,MAE,MSE,PSNR,SSIM
body_part,,,,
brain,150.888 +- 28.8315,1566.74 +- 151.691,40.1822 +- 0.485994,0.756722 +- 0.0374324


## Pelvis


In [44]:
csv_path = Path("/local/scratch/datasets/FullbodySCT/Synthrad_combined_preprocessed/100results/cut_synthrad_pelvis_final/test_50/test_metrics_over_all.csv")
df = load_metrics_csv(csv_path)

# show_volume_slice_influence(df, volume=3)


In [45]:
# Run metrics for the selected CSV path

csv_path = Path("/local/scratch/datasets/FullbodySCT/Synthrad_combined_preprocessed/100results/cut_synthrad_pelvis_final/test_50/test_metrics_over_all.csv")
show_metrics_for_csv(csv_path)


Slice metrics | non-finite values ignored in stats -> PSNR: 1
Overall metrics (slice-based):


,mean +- std
MAE,97.3549 +- 66.2945
MSE,2447.62 +- 486.721
PSNR,38.4419 +- 0.87462
SSIM,0.750303 +- 0.0881869


Metrics by body part (slice-based):


,MAE,MSE,PSNR,SSIM
body_part,,,,
pelvis,97.3549 +- 66.2945,2447.62 +- 486.721,38.4419 +- 0.87462,0.750303 +- 0.0881869


Overall metrics (volume-based):


,mean +- std
MAE,95.9329 +- 16.9005
MSE,2421.15 +- 277.17
PSNR,38.4709 +- 0.516723
SSIM,0.750814 +- 0.0344813


Metrics by body part (volume-based):


,MAE,MSE,PSNR,SSIM
body_part,,,,
pelvis,95.9329 +- 16.9005,2421.15 +- 277.17,38.4709 +- 0.516723,0.750814 +- 0.0344813


## Thorax


In [46]:
csv_path = Path("/local/scratch/datasets/FullbodySCT/Synthrad_combined_preprocessed/100results/cut_synthrad_TH_final/test_50/test_metrics_over_all.csv")
df = load_metrics_csv(csv_path)

# show_volume_slice_influence(df, volume=3)


In [47]:
# Run metrics for the selected CSV path

csv_path = Path("/local/scratch/datasets/FullbodySCT/Synthrad_combined_preprocessed/100results/cut_synthrad_TH_final/test_50/test_metrics_over_all.csv")
show_metrics_for_csv(csv_path)


Slice metrics | non-finite values ignored in stats -> MAE: 84, MSE: 84, PSNR: 87, SSIM: 84
Overall metrics (slice-based):


,mean +- std
MAE,190.791 +- 283.109
MSE,3865.51 +- 2194.15
PSNR,36.6298 +- 1.51162
SSIM,0.642609 +- 0.137325


Metrics by body part (slice-based):


,MAE,MSE,PSNR,SSIM
body_part,,,,
TH,190.791 +- 283.109,3865.51 +- 2194.15,36.6298 +- 1.51162,0.642609 +- 0.137325


Overall metrics (volume-based):


,mean +- std
MAE,155.654 +- 38.3489
MSE,3709.93 +- 614.938
PSNR,36.7133 +- 0.680291
SSIM,0.657615 +- 0.0486513


Metrics by body part (volume-based):


,MAE,MSE,PSNR,SSIM
body_part,,,,
TH,155.654 +- 38.3489,3709.93 +- 614.938,36.7133 +- 0.680291,0.657615 +- 0.0486513


## Head and Neck


In [48]:
csv_path = Path("/local/scratch/datasets/FullbodySCT/Synthrad_combined_preprocessed/100results/cut_synthrad_HN_final/test_50/test_metrics_over_all.csv")
df = load_metrics_csv(csv_path)

# show_volume_slice_influence(df, volume=3)


In [49]:
# Run metrics for the selected CSV path

csv_path = Path("/local/scratch/datasets/FullbodySCT/Synthrad_combined_preprocessed/100results/cut_synthrad_HN_final/test_50/test_metrics_over_all.csv")
show_metrics_for_csv(csv_path)


Slice metrics | non-finite values ignored in stats -> MAE: 79, MSE: 79, PSNR: 247, SSIM: 79
Overall metrics (slice-based):


,mean +- std
MAE,280.539 +- 428.507
MSE,1092.07 +- 7665.41
PSNR,37.965 +- 1.94513
SSIM,0.600698 +- 0.175513


Metrics by body part (slice-based):


,MAE,MSE,PSNR,SSIM
body_part,,,,
HN,280.539 +- 428.507,1092.07 +- 7665.41,37.965 +- 1.94513,0.600698 +- 0.175513


Overall metrics (volume-based):


,mean +- std
MAE,310.551 +- 165.448
MSE,896.636 +- 1543.59
PSNR,35.3144 +- 2.11648
SSIM,0.582894 +- 0.0933291


Metrics by body part (volume-based):


,MAE,MSE,PSNR,SSIM
body_part,,,,
HN,310.551 +- 165.448,896.636 +- 1543.59,35.3144 +- 2.11648,0.582894 +- 0.0933291


## Fullbody


In [50]:
csv_path = Path("/local/scratch/datasets/FullbodySCT/Synthrad_combined_preprocessed/100results/cut_synthrad_allregions_final/test_50/test_metrics_over_all.csv")
df = load_metrics_csv(csv_path)

# show_volume_slice_influence(df, volume=3)


In [51]:
# Run metrics for the selected CSV path

csv_path = Path("/local/scratch/datasets/FullbodySCT/Synthrad_combined_preprocessed/100results/cut_synthrad_allregions_final/test_50/test_metrics_over_all.csv")
show_metrics_for_csv(csv_path)


Slice metrics | non-finite values ignored in stats -> MAE: 218, MSE: 218, PSNR: 368, SSIM: 218
Overall metrics (slice-based):


,mean +- std
MAE,181.106 +- 281.175
MSE,3235.65 +- 3931.77
PSNR,38.0827 +- 2.54779
SSIM,0.677079 +- 0.147906


Metrics by body part (slice-based):


,MAE,MSE,PSNR,SSIM
body_part,,,,
AB,171.829 +- 325.449,4681.06 +- 4522.03,36.2439 +- 2.08768,0.616842 +- 0.123862
HN,262.587 +- 429.41,4131.63 +- 6086.67,37.6178 +- 3.03195,0.63673 +- 0.186562
TH,196.057 +- 337.632,4267.3 +- 4766.45,36.7409 +- 2.07279,0.649473 +- 0.138453
brain,176.887 +- 146.08,1626.01 +- 858.331,40.2962 +- 1.73057,0.726881 +- 0.144994
pelvis,106.113 +- 85.2909,2803.76 +- 545.851,37.8247 +- 0.832363,0.71202 +- 0.0937712


Overall metrics (volume-based):


,mean +- std
MAE,176.534 +- 110.666
MSE,3416.89 +- 1321.13
PSNR,37.5913 +- 1.60083
SSIM,0.668699 +- 0.0748214


Metrics by body part (volume-based):


,MAE,MSE,PSNR,SSIM
body_part,,,,
AB,151.875 +- 61.4835,4540.76 +- 1299.48,36.1237 +- 1.0535,0.622502 +- 0.040727
HN,294.985 +- 166.617,4287.15 +- 879.144,37.0501 +- 1.17282,0.615758 +- 0.0965054
TH,154.143 +- 42.4856,3723.3 +- 533.501,36.8935 +- 0.503295,0.663964 +- 0.0455308
brain,151.156 +- 27.5659,1558.92 +- 144.747,40.2233 +- 0.48902,0.743843 +- 0.0268678
pelvis,104.188 +- 19.174,2780.94 +- 278.346,37.7863 +- 0.395159,0.709191 +- 0.0294098


# Pix2Pix


## Abdomen


In [52]:
csv_path = Path("/local/scratch/datasets/FullbodySCT/Synthrad_combined_preprocessed/100results/pix2pix_synthrad_abdomen_final/test_50/test_metrics_over_all.csv")
df = load_metrics_csv(csv_path)

# show_volume_slice_influence(df, volume=3)


In [53]:
# Run metrics for the selected CSV path

csv_path = Path("/local/scratch/datasets/FullbodySCT/Synthrad_combined_preprocessed/100results/pix2pix_synthrad_abdomen_final/test_50/test_metrics_over_all.csv")
show_metrics_for_csv(csv_path)


Slice metrics | non-finite values ignored in stats -> MAE: 55, MSE: 55, PSNR: 72, SSIM: 55
Overall metrics (slice-based):


,mean +- std
MAE,119.112 +- 61.9466
MSE,3020.88 +- 760.462
PSNR,37.5801 +- 1.42623
SSIM,0.640225 +- 0.0838694


Metrics by body part (slice-based):


,MAE,MSE,PSNR,SSIM
body_part,,,,
AB,119.112 +- 61.9466,3020.88 +- 760.462,37.5801 +- 1.42623,0.640225 +- 0.0838694


Overall metrics (volume-based):


,mean +- std
MAE,116.864 +- 35.338
MSE,3006.09 +- 407.485
PSNR,37.3111 +- 1.27225
SSIM,0.643834 +- 0.0490444


Metrics by body part (volume-based):


,MAE,MSE,PSNR,SSIM
body_part,,,,
AB,116.864 +- 35.338,3006.09 +- 407.485,37.3111 +- 1.27225,0.643834 +- 0.0490444


## Brain


In [54]:
csv_path = Path("/local/scratch/datasets/FullbodySCT/Synthrad_combined_preprocessed/100results/pix2pix_synthrad_brain_final/test_50/test_metrics_over_all.csv")
df = load_metrics_csv(csv_path)

# show_volume_slice_influence(df, volume=3)


In [55]:
# Run metrics for the selected CSV path

csv_path = Path("/local/scratch/datasets/FullbodySCT/Synthrad_combined_preprocessed/100results/pix2pix_synthrad_brain_final/test_50/test_metrics_over_all.csv")
show_metrics_for_csv(csv_path)


Slice metrics | non-finite values ignored in stats -> PSNR: 26
Overall metrics (slice-based):


,mean +- std
MAE,156.872 +- 107.184
MSE,1903.64 +- 703.388
PSNR,39.6838 +- 1.55021
SSIM,0.733431 +- 0.157798


Metrics by body part (slice-based):


,MAE,MSE,PSNR,SSIM
body_part,,,,
brain,156.872 +- 107.184,1903.64 +- 703.388,39.6838 +- 1.55021,0.733431 +- 0.157798


Overall metrics (volume-based):


,mean +- std
MAE,140.871 +- 24.5285
MSE,1773.69 +- 141.158
PSNR,39.8552 +- 0.432344
SSIM,0.749878 +- 0.0399167


Metrics by body part (volume-based):


,MAE,MSE,PSNR,SSIM
body_part,,,,
brain,140.871 +- 24.5285,1773.69 +- 141.158,39.8552 +- 0.432344,0.749878 +- 0.0399167


## Pelvis


In [56]:
csv_path = Path("/local/scratch/datasets/FullbodySCT/Synthrad_combined_preprocessed/100results/pix2pix_synthrad_pelvis_final/test_50/test_metrics_over_all.csv")
df = load_metrics_csv(csv_path)

# show_volume_slice_influence(df, volume=3)


In [57]:
# Run metrics for the selected CSV path

csv_path = Path("/local/scratch/datasets/FullbodySCT/Synthrad_combined_preprocessed/100results/pix2pix_synthrad_pelvis_final/test_50/test_metrics_over_all.csv")
show_metrics_for_csv(csv_path)


Overall metrics (slice-based):


,mean +- std
MAE,85.3043 +- 32.2833
MSE,2423.99 +- 588.517
PSNR,38.5153 +- 0.982287
SSIM,0.764115 +- 0.0836161


Metrics by body part (slice-based):


,MAE,MSE,PSNR,SSIM
body_part,,,,
pelvis,85.3043 +- 32.2833,2423.99 +- 588.517,38.5153 +- 0.982287,0.764115 +- 0.0836161


Overall metrics (volume-based):


,mean +- std
MAE,83.9193 +- 14.3518
MSE,2401.24 +- 281.555
PSNR,38.5438 +- 0.476474
SSIM,0.763569 +- 0.0324483


Metrics by body part (volume-based):


,MAE,MSE,PSNR,SSIM
body_part,,,,
pelvis,83.9193 +- 14.3518,2401.24 +- 281.555,38.5438 +- 0.476474,0.763569 +- 0.0324483


## Thorax


In [58]:
csv_path = Path("/local/scratch/datasets/FullbodySCT/Synthrad_combined_preprocessed/100results/pix2pix_synthrad_thorax_final/test_50/test_metrics_over_all.csv")
df = load_metrics_csv(csv_path)

# show_volume_slice_influence(df, volume=3)


In [59]:
# Run metrics for the selected CSV path

csv_path = Path("/local/scratch/datasets/FullbodySCT/Synthrad_combined_preprocessed/100results/pix2pix_synthrad_thorax_final/test_50/test_metrics_over_all.csv")
show_metrics_for_csv(csv_path)


Slice metrics | non-finite values ignored in stats -> MAE: 84, MSE: 84, PSNR: 84, SSIM: 84
Overall metrics (slice-based):


,mean +- std
MAE,132.449 +- 50.9977
MSE,2963.88 +- 673.259
PSNR,37.7606 +- 2.11463
SSIM,0.671091 +- 0.0822805


Metrics by body part (slice-based):


,MAE,MSE,PSNR,SSIM
body_part,,,,
TH,132.449 +- 50.9977,2963.88 +- 673.259,37.7606 +- 2.11463,0.671091 +- 0.0822805


Overall metrics (volume-based):


,mean +- std
MAE,133.603 +- 29.0347
MSE,3010.69 +- 216.413
PSNR,37.5766 +- 0.351578
SSIM,0.666497 +- 0.0471883


Metrics by body part (volume-based):


,MAE,MSE,PSNR,SSIM
body_part,,,,
TH,133.603 +- 29.0347,3010.69 +- 216.413,37.5766 +- 0.351578,0.666497 +- 0.0471883


## Head and Neck


In [60]:
csv_path = Path("/local/scratch/datasets/FullbodySCT/Synthrad_combined_preprocessed/100results/pix2pix_synthrad_headneck_final/test_50/test_metrics_over_all.csv")
df = load_metrics_csv(csv_path)

# show_volume_slice_influence(df, volume=3)


In [61]:
# Run metrics for the selected CSV path

csv_path = Path("/local/scratch/datasets/FullbodySCT/Synthrad_combined_preprocessed/100results/pix2pix_synthrad_headneck_final/test_50/test_metrics_over_all.csv")
show_metrics_for_csv(csv_path)


Slice metrics | non-finite values ignored in stats -> MAE: 79, MSE: 79, PSNR: 101, SSIM: 79
Overall metrics (slice-based):


,mean +- std
MAE,145.765 +- 123.56
MSE,2245.6 +- 760.676
PSNR,38.9892 +- 1.79017
SSIM,0.690593 +- 0.1216


Metrics by body part (slice-based):


,MAE,MSE,PSNR,SSIM
body_part,,,,
HN,145.765 +- 123.56,2245.6 +- 760.676,38.9892 +- 1.79017,0.690593 +- 0.1216


Overall metrics (volume-based):


,mean +- std
MAE,161.377 +- 78.08
MSE,2278.24 +- 234.05
PSNR,38.4551 +- 1.03733
SSIM,0.667675 +- 0.0827526


Metrics by body part (volume-based):


,MAE,MSE,PSNR,SSIM
body_part,,,,
HN,161.377 +- 78.08,2278.24 +- 234.05,38.4551 +- 1.03733,0.667675 +- 0.0827526


## Fullbody


In [62]:
csv_path = Path("/local/scratch/datasets/FullbodySCT/Synthrad_combined_preprocessed/100results/pix2pix_synthrad_allregion_final/test_50/test_metrics_over_all.csv")
df = load_metrics_csv(csv_path)

# show_volume_slice_influence(df, volume=3)


In [63]:
# Run metrics for the selected CSV path

csv_path = Path("/local/scratch/datasets/FullbodySCT/Synthrad_combined_preprocessed/100results/pix2pix_synthrad_allregion_final/test_50/test_metrics_over_all.csv")
show_metrics_for_csv(csv_path)


Slice metrics | non-finite values ignored in stats -> MAE: 218, MSE: 218, PSNR: 306, SSIM: 218
Overall metrics (slice-based):


,mean +- std
MAE,137.818 +- 98.0656
MSE,2435.61 +- 887.126
PSNR,38.6913 +- 1.97107
SSIM,0.697036 +- 0.127812


Metrics by body part (slice-based):


,MAE,MSE,PSNR,SSIM
body_part,,,,
AB,114.844 +- 59.7017,3015.94 +- 797.767,37.5891 +- 1.45025,0.651676 +- 0.0768542
HN,160.143 +- 148.108,2402.99 +- 925.546,38.7477 +- 2.00545,0.67437 +- 0.128345
TH,138.832 +- 57.0056,2984.19 +- 697.583,37.7224 +- 2.19569,0.665076 +- 0.0924517
brain,159.6 +- 97.5481,1855.21 +- 783.023,39.8831 +- 1.78641,0.722834 +- 0.166275
pelvis,100.416 +- 80.8289,2393.77 +- 532.591,38.5618 +- 1.13311,0.745996 +- 0.085726


Volume metrics | non-finite values ignored in stats -> PSNR: 1
Overall metrics (volume-based):


,mean +- std
MAE,137.226 +- 59.9462
MSE,2518.75 +- 549.837
PSNR,38.2594 +- 1.36866
SSIM,0.687213 +- 0.0686291


Metrics by body part (volume-based):


,MAE,MSE,PSNR,SSIM
body_part,,,,
AB,113.579 +- 34.5321,3017.85 +- 370.731,37.2591 +- 1.35659,0.653745 +- 0.0491429
HN,180.481 +- 98.3176,2440.42 +- 299.883,37.967 +- 1.37521,0.650649 +- 0.0831836
TH,139.53 +- 33.9458,3041.74 +- 276.803,37.4927 +- 0.433568,0.66009 +- 0.0544941
brain,144.237 +- 16.6608,1727.66 +- 160.98,40.0951 +- 0.385306,0.738146 +- 0.0338189
pelvis,98.694 +- 16.6353,2383.49 +- 240.461,38.5197 +- 0.460579,0.741563 +- 0.0305146


# CycleGAN


## Abdomen


In [64]:
csv_path = Path("/local/scratch/datasets/FullbodySCT/Synthrad_combined_preprocessed/100results/cyclegan_abdomen_final/test_50/test_metrics_over_all.csv")
df = load_metrics_csv(csv_path)

# show_volume_slice_influence(df, volume=3)


In [65]:
# Run metrics for the selected CSV path

csv_path = Path("/local/scratch/datasets/FullbodySCT/Synthrad_combined_preprocessed/100results/cyclegan_abdomen_final/test_50/test_metrics_over_all.csv")
show_metrics_for_csv(csv_path)


Slice metrics | non-finite values ignored in stats -> MAE: 55, MSE: 55, PSNR: 81, SSIM: 55
Overall metrics (slice-based):


,mean +- std
MAE,178.264 +- 112.808
MSE,4390.18 +- 1341.65
PSNR,36.0748 +- 2.0266
SSIM,0.521518 +- 0.110381


Metrics by body part (slice-based):


,MAE,MSE,PSNR,SSIM
body_part,,,,
AB,178.264 +- 112.808,4390.18 +- 1341.65,36.0748 +- 2.0266,0.521518 +- 0.110381


Overall metrics (volume-based):


,mean +- std
MAE,174.472 +- 54.3367
MSE,4467.67 +- 597.615
PSNR,35.5683 +- 1.50013
SSIM,0.527928 +- 0.0819799


Metrics by body part (volume-based):


,MAE,MSE,PSNR,SSIM
body_part,,,,
AB,174.472 +- 54.3367,4467.67 +- 597.615,35.5683 +- 1.50013,0.527928 +- 0.0819799


## Brain


In [66]:
csv_path = Path("/local/scratch/datasets/FullbodySCT/Synthrad_combined_preprocessed/100results/cyclegan_brain_final/test_50/test_metrics_over_all.csv")
df = load_metrics_csv(csv_path)

# show_volume_slice_influence(df, volume=3)


In [67]:
# Run metrics for the selected CSV path

csv_path = Path("/local/scratch/datasets/FullbodySCT/Synthrad_combined_preprocessed/100results/cyclegan_brain_final/test_50/test_metrics_over_all.csv")
show_metrics_for_csv(csv_path)


Slice metrics | non-finite values ignored in stats -> PSNR: 84
Overall metrics (slice-based):


,mean +- std
MAE,301.096 +- 126.009
MSE,1812.12 +- 737.071
PSNR,39.8439 +- 1.72628
SSIM,0.517178 +- 0.167235


Metrics by body part (slice-based):


,MAE,MSE,PSNR,SSIM
body_part,,,,
brain,301.096 +- 126.009,1812.12 +- 737.071,39.8439 +- 1.72628,0.517178 +- 0.167235


Overall metrics (volume-based):


,mean +- std
MAE,298.628 +- 58.686
MSE,1761.76 +- 188.451
PSNR,39.7333 +- 0.688135
SSIM,0.500211 +- 0.103808


Metrics by body part (volume-based):


,MAE,MSE,PSNR,SSIM
body_part,,,,
brain,298.628 +- 58.686,1761.76 +- 188.451,39.7333 +- 0.688135,0.500211 +- 0.103808


## Pelvis


In [68]:
csv_path = Path("/local/scratch/datasets/FullbodySCT/Synthrad_combined_preprocessed/100results/cyclegan_pelvis_final/test_50/test_metrics_over_all.csv")
df = load_metrics_csv(csv_path)

# show_volume_slice_influence(df, volume=3)


In [69]:
# Run metrics for the selected CSV path

csv_path = Path("/local/scratch/datasets/FullbodySCT/Synthrad_combined_preprocessed/100results/cyclegan_pelvis_final/test_50/test_metrics_over_all.csv")
show_metrics_for_csv(csv_path)


Overall metrics (slice-based):


,mean +- std
MAE,195.527 +- 88.7502
MSE,5180.7 +- 1113.41
PSNR,35.2075 +- 0.980212
SSIM,0.525097 +- 0.072071


Metrics by body part (slice-based):


,MAE,MSE,PSNR,SSIM
body_part,,,,
pelvis,195.527 +- 88.7502,5180.7 +- 1113.41,35.2075 +- 0.980212,0.525097 +- 0.072071


Overall metrics (volume-based):


,mean +- std
MAE,208.604 +- 91.4367
MSE,5029.67 +- 969.383
PSNR,35.343 +- 0.869113
SSIM,0.529231 +- 0.0438917


Metrics by body part (volume-based):


,MAE,MSE,PSNR,SSIM
body_part,,,,
pelvis,208.604 +- 91.4367,5029.67 +- 969.383,35.343 +- 0.869113,0.529231 +- 0.0438917


## Thorax


In [70]:
csv_path = Path("/local/scratch/datasets/FullbodySCT/Synthrad_combined_preprocessed/100results/cyclegan_thorax_final/test_50/test_metrics_over_all.csv")
df = load_metrics_csv(csv_path)

# show_volume_slice_influence(df, volume=3)


In [71]:
# Run metrics for the selected CSV path

csv_path = Path("/local/scratch/datasets/FullbodySCT/Synthrad_combined_preprocessed/100results/cyclegan_thorax_final/test_50/test_metrics_over_all.csv")
show_metrics_for_csv(csv_path)


Slice metrics | non-finite values ignored in stats -> MAE: 84, MSE: 84, PSNR: 88, SSIM: 84
Overall metrics (slice-based):


,mean +- std
MAE,225.314 +- 125.209
MSE,3115.16 +- 873.977
PSNR,37.5564 +- 1.71883
SSIM,0.579952 +- 0.142547


Metrics by body part (slice-based):


,MAE,MSE,PSNR,SSIM
body_part,,,,
TH,225.314 +- 125.209,3115.16 +- 873.977,37.5564 +- 1.71883,0.579952 +- 0.142547


Volume metrics | non-finite values ignored in stats -> PSNR: 1
Overall metrics (volume-based):


,mean +- std
MAE,223.936 +- 91.9382
MSE,3152.49 +- 396.07
PSNR,37.3988 +- 0.553902
SSIM,0.578733 +- 0.108016


Metrics by body part (volume-based):


,MAE,MSE,PSNR,SSIM
body_part,,,,
TH,223.936 +- 91.9382,3152.49 +- 396.07,37.3988 +- 0.553902,0.578733 +- 0.108016


## Head and Neck


In [72]:
csv_path = Path("/local/scratch/datasets/FullbodySCT/Synthrad_combined_preprocessed/100results/cyclegan_head_neck_final/test_50/test_metrics_over_all.csv")
df = load_metrics_csv(csv_path)

# show_volume_slice_influence(df, volume=3)


In [73]:
# Run metrics for the selected CSV path

csv_path = Path("/local/scratch/datasets/FullbodySCT/Synthrad_combined_preprocessed/100results/cyclegan_head_neck_final/test_50/test_metrics_over_all.csv")
show_metrics_for_csv(csv_path)


Slice metrics | non-finite values ignored in stats -> MAE: 79, MSE: 79, PSNR: 109, SSIM: 79
Overall metrics (slice-based):


,mean +- std
MAE,226.898 +- 163.14
MSE,2758.62 +- 1080.99
PSNR,38.2504 +- 2.35065
SSIM,0.55884 +- 0.128493


Metrics by body part (slice-based):


,MAE,MSE,PSNR,SSIM
body_part,,,,
HN,226.898 +- 163.14,2758.62 +- 1080.99,38.2504 +- 2.35065,0.55884 +- 0.128493


Overall metrics (volume-based):


,mean +- std
MAE,243.337 +- 102.645
MSE,2748.08 +- 425.628
PSNR,37.6271 +- 1.3204
SSIM,0.543353 +- 0.0756491


Metrics by body part (volume-based):


,MAE,MSE,PSNR,SSIM
body_part,,,,
HN,243.337 +- 102.645,2748.08 +- 425.628,37.6271 +- 1.3204,0.543353 +- 0.0756491


## Fullbody


In [74]:
csv_path = Path("/local/scratch/datasets/FullbodySCT/Synthrad_combined_preprocessed/100results/cyclegan_allregions_final/test_50/test_metrics_over_all.csv")
df = load_metrics_csv(csv_path)

# show_volume_slice_influence(df, volume=3)


In [75]:
# Run metrics for the selected CSV path

csv_path = Path("/local/scratch/datasets/FullbodySCT/Synthrad_combined_preprocessed/100results/cyclegan_allregions_final/test_50/test_metrics_over_all.csv")
show_metrics_for_csv(csv_path)


Slice metrics | non-finite values ignored in stats -> MAE: 218, MSE: 218, PSNR: 363, SSIM: 218
Overall metrics (slice-based):


,mean +- std
MAE,266.775 +- 140.064
MSE,2949.94 +- 1439.7
PSNR,38.2021 +- 2.81707
SSIM,0.538017 +- 0.138173


Metrics by body part (slice-based):


,MAE,MSE,PSNR,SSIM
body_part,,,,
AB,236.676 +- 136.414,3913.5 +- 1217.07,36.5879 +- 2.13766,0.511857 +- 0.123491
HN,258.368 +- 161.621,2597.74 +- 1163.81,38.5561 +- 2.35306,0.530857 +- 0.125814
TH,261.902 +- 142.37,3340.38 +- 936.999,37.2935 +- 1.93715,0.550422 +- 0.158763
brain,315.515 +- 116.943,1631.17 +- 836.703,40.6473 +- 2.40911,0.52221 +- 0.150404
pelvis,225.386 +- 130.929,4235.91 +- 1024.12,36.1205 +- 1.79535,0.582515 +- 0.103954


Volume metrics | non-finite values ignored in stats -> PSNR: 1
Overall metrics (volume-based):


,mean +- std
MAE,257.282 +- 89.6774
MSE,3131.83 +- 1079.73
PSNR,37.5081 +- 2.04327
SSIM,0.537648 +- 0.0850759


Metrics by body part (volume-based):


,MAE,MSE,PSNR,SSIM
body_part,,,,
AB,223.947 +- 99.3627,4012.54 +- 554.37,36.0699 +- 1.44705,0.522159 +- 0.0991575
HN,271.248 +- 95.8539,2608.41 +- 463.675,37.8894 +- 1.07466,0.521134 +- 0.0598953
TH,258.209 +- 110.93,3435.85 +- 461.909,37.0446 +- 0.64929,0.550936 +- 0.127377
brain,302.034 +- 28.8624,1520.28 +- 170.793,40.7584 +- 0.818231,0.523556 +- 0.0481865
pelvis,227.871 +- 67.8463,4198.39 +- 579.231,35.6761 +- 0.852382,0.574127 +- 0.0617434
